# Car Resale Data Pipeline (Medallion Architecture)

Raw car resale listings are cleaned and enriched through Bronze -> Silver -> Gold
layers, ending in a dataset ready for pricing analysis and dashboarding.
Built as a hands-on task for PwC's Jump Start Your Career programme (Data
Engineering track). See README.md for the full business requirements this
was built against.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import datetime

# Bronze Layer
### Raw Data ingestion

In [2]:
df = pd.read_csv('data/car_resale_prices.csv')
print(df.shape)
print(df.columns)

(17446, 15)
Index(['Unnamed: 0', 'full_name', 'resale_price', 'registered_year',
       'engine_capacity', 'insurance', 'transmission_type', 'kms_driven',
       'owner_type', 'fuel_type', 'max_power', 'seats', 'mileage', 'body_type',
       'city'],
      dtype='object')


In [3]:
# the source file's index was exported as an unnamed column - promote it to
# the actual DataFrame index instead of treating it as a data column
df_bronze_layer = df.rename(columns={'Unnamed: 0': 'index'}).set_index('index')
df_bronze_layer.to_csv('data/car_bronze_layer.csv', index=True)
df_bronze_layer.head()

,full_name,resale_price,registered_year,engine_capacity,insurance,transmission_type,kms_driven,owner_type,fuel_type,max_power,seats,mileage,body_type,city
index,,,,,,,,,,,,,,
0,2017 Maruti Baleno 1.2 Alpha,₹ 5.45 Lakh,2017,1197 cc,Third Party insurance,Manual,"40,000 Kms",First Owner,Petrol,83.1bhp,5.0,21.4 kmpl,Hatchback,Agra
1,2018 Tata Hexa XTA,₹ 10 Lakh,2018,2179 cc,Third Party insurance,Automatic,"70,000 Kms",First Owner,Diesel,153.86bhp,7.0,17.6 kmpl,MUV,Agra
2,2015 Maruti Swift Dzire VXI,₹ 4.50 Lakh,2015,1197 cc,Third Party insurance,Manual,"70,000 Kms",Second Owner,Petrol,83.14bhp,5.0,20.85 kmpl,Sedan,Agra
3,2015 Maruti Swift Dzire VXI,₹ 4.50 Lakh,2015,1197 cc,Third Party insurance,Manual,"70,000 Kms",Second Owner,Petrol,83.14bhp,5.0,20.85 kmpl,Sedan,Agra
4,2009 Hyundai i10 Magna 1.1,₹ 1.60 Lakh,2009,1086 cc,Third Party insurance,Manual,"80,000 Kms",First Owner,Petrol,68.05bhp,5.0,19.81 kmpl,Hatchback,Agra


# Silver Layer
### Data cleaning to ensure consistent formatting:
1. Removing null rows
2. Handling duplicate rows
3. Standardizing column names
4. Handling missing values
5. Correcting data types & fixing inconsistent formatting
6. Removing and handling outliers

In [4]:
print(df_bronze_layer.shape)
print(df_bronze_layer.columns)

(17446, 14)
Index(['full_name', 'resale_price', 'registered_year', 'engine_capacity',
       'insurance', 'transmission_type', 'kms_driven', 'owner_type',
       'fuel_type', 'max_power', 'seats', 'mileage', 'body_type', 'city'],
      dtype='object')


### 1) Drop all rows with NULL values in all columns

In [5]:
df_bronze_layer = df_bronze_layer.dropna(how='all')
df_bronze_layer.shape

(17446, 14)

### 2) Handling duplicate rows

In [6]:
num_duplicate_rows = df_bronze_layer.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicate_rows}")
print(f"% of duplicated rows: {num_duplicate_rows / len(df_bronze_layer)}")

Number of duplicate rows: 203
% of duplicated rows: 0.01163590507852803


Drop duplicate rows since they are exact matches across all columns and their percentage does not exceed 5% of the dataset

In [7]:
df_bronze_layer.drop_duplicates(inplace=True)
df_bronze_layer.shape

(17243, 14)

### 3) Standardize column names:  
Column name standardization to ensure consistency and make them easier to work with

In [8]:
df_bronze_layer.columns = (
    df_bronze_layer.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'\W', '', regex=True)
)
print(df_bronze_layer.columns)

Index(['full_name', 'resale_price', 'registered_year', 'engine_capacity',
       'insurance', 'transmission_type', 'kms_driven', 'owner_type',
       'fuel_type', 'max_power', 'seats', 'mileage', 'body_type', 'city'],
      dtype='object')


### 4) Handling missing values

In [9]:
# only inspect low-cardinality columns (< 20 unique values) - these are the
# categorical ones where printing every unique value is actually readable
for column in df_bronze_layer.columns:
    if df_bronze_layer[column].nunique() < 20:
        null_count = df_bronze_layer[column].isnull().sum()
        print(f"There are {null_count} null values in this column")
        print(f"Unique values in '{column}' column:")
        print(df_bronze_layer[column].unique())
        print('-' * 50)

There are 7 null values in this column
Unique values in 'insurance' column:
['Third Party insurance' 'Comprehensive' 'Zero Dep' 'Third Party'
 'Not Available' nan '2' '1']
--------------------------------------------------
There are 0 null values in this column
Unique values in 'transmission_type' column:
['Manual' 'Automatic']
--------------------------------------------------
There are 45 null values in this column
Unique values in 'owner_type' column:
['First Owner' 'Second Owner' 'Third Owner' 'Fifth Owner' 'Fourth Owner'
 nan]
--------------------------------------------------
There are 0 null values in this column
Unique values in 'fuel_type' column:
['Petrol' 'Diesel' 'CNG' 'Electric' 'LPG']
--------------------------------------------------
There are 10 null values in this column
Unique values in 'seats' column:
[ 5.  7.  8.  6.  4.  9.  2. nan 10. 14.]
--------------------------------------------------
There are 0 null values in this column
Unique values in 'city' column:
['Ag

In [10]:
df_bronze_layer['insurance'] = df_bronze_layer['insurance'].replace('Not Available', 'Unknown')
df_bronze_layer['insurance'] = df_bronze_layer['insurance'].fillna('Unknown')

In [11]:
print(df_bronze_layer.isnull().mean() * 100)

full_name            0.000000
resale_price         0.000000
registered_year      0.394363
engine_capacity      0.075393
insurance            0.000000
transmission_type    0.000000
kms_driven           0.017398
owner_type           0.260975
fuel_type            0.000000
max_power            0.579945
seats                0.057995
mileage              2.911326
body_type            0.000000
city                 0.000000
dtype: float64


Since the percentage of missing values in all features does not exceed 5%, it is more appropriate to impute the missing values instead of dropping them. We will use the median for numerical columns to avoid the impact of outliers, and the mode for categorical columns—except for the 'insurance' column, which contains 'Not Available' entries. For consistency, both missing and 'Not Available' values in this column will be replaced with an 'Unknown' label to preserve a common category

In [12]:
numerical_cols = df_bronze_layer.select_dtypes(include=['number']).columns
categorical_cols = df_bronze_layer.select_dtypes(include=['object', 'string']).columns

for col in numerical_cols:
    if df_bronze_layer[col].isnull().sum() > 0:
        median_value = df_bronze_layer[col].median()
        df_bronze_layer[col] = df_bronze_layer[col].fillna(median_value)

for col in categorical_cols:
    if col != 'insurance' and df_bronze_layer[col].isnull().sum() > 0:
        mode_value = df_bronze_layer[col].mode()[0]
        df_bronze_layer[col] = df_bronze_layer[col].fillna(mode_value)

for col in categorical_cols:
    df_bronze_layer[col] = df_bronze_layer[col].astype("string")

In [13]:
print(df_bronze_layer.isnull().mean() * 100)
df_bronze_layer.info()

full_name            0.0
resale_price         0.0
registered_year      0.0
engine_capacity      0.0
insurance            0.0
transmission_type    0.0
kms_driven           0.0
owner_type           0.0
fuel_type            0.0
max_power            0.0
seats                0.0
mileage              0.0
body_type            0.0
city                 0.0
dtype: float64
<class 'pandas.core.frame.DataFrame'>
Index: 17243 entries, 0 to 17445
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   full_name          17243 non-null  string 
 1   resale_price       17243 non-null  string 
 2   registered_year    17243 non-null  string 
 3   engine_capacity    17243 non-null  string 
 4   insurance          17243 non-null  string 
 5   transmission_type  17243 non-null  string 
 6   kms_driven         17243 non-null  string 
 7   owner_type         17243 non-null  string 
 8   fuel_type          17243 non-null  string 
 9 

### 5) Correcting data types & fixing inconsistent formatting

In [14]:
for column in df_bronze_layer.columns:
    if df_bronze_layer[column].nunique() < 20:
        print(f"Unique values in '{column}' column:")
        print(df_bronze_layer[column].unique())
        print('-' * 50)

Unique values in 'insurance' column:
<StringArray>
['Third Party insurance',         'Comprehensive',              'Zero Dep',
           'Third Party',               'Unknown',                     '2',
                     '1']
Length: 7, dtype: string
--------------------------------------------------
Unique values in 'transmission_type' column:
<StringArray>
['Manual', 'Automatic']
Length: 2, dtype: string
--------------------------------------------------
Unique values in 'owner_type' column:
<StringArray>
['First Owner', 'Second Owner', 'Third Owner', 'Fifth Owner', 'Fourth Owner']
Length: 5, dtype: string
--------------------------------------------------
Unique values in 'fuel_type' column:
<StringArray>
['Petrol', 'Diesel', 'CNG', 'Electric', 'LPG']
Length: 5, dtype: string
--------------------------------------------------
Unique values in 'seats' column:
[ 5.  7.  8.  6.  4.  9.  2. 10. 14.]
--------------------------------------------------
Unique values in 'city' column:
<S

In [15]:
# Fixed: the '2' -> 'Comprehensive ' mapping had a trailing space, which would
# have created a second, near-duplicate 'Comprehensive ' category alongside
# the existing 'Comprehensive' entries instead of merging into it
df_bronze_layer['insurance'] = df_bronze_layer['insurance'].replace(
    {'1': 'Third Party', '2': 'Comprehensive', 'Third Party insurance': 'Third Party'}
)

Standardized Full Name:

Parse full_name to separate year, make, model, and variant; store components separately

In [16]:
# Note: this assumes make and model are always exactly one token each
# (e.g. "2017 Maruti Baleno 1.2 Alpha" -> make=Maruti, model=Baleno). Multi-word
# makes/models (e.g. "Land Rover", "Mercedes Benz") would be mis-split - see
# README for why this wasn't fixed here.
def norm_full(full_name):
    parts = full_name.split()
    year = parts[0]
    make = parts[1]
    model = parts[2]
    variant = ' '.join(parts[3:]) if len(parts) > 3 else ''
    return pd.Series([year, make, model, variant], dtype="string")

df_bronze_layer[['year', 'make', 'model', 'variant']] = df_bronze_layer['full_name'].apply(norm_full)
df_bronze_layer['year'] = df_bronze_layer['year'].astype(int)
df_bronze_layer['make'] = df_bronze_layer['make'].astype("string")
df_bronze_layer['model'] = df_bronze_layer['model'].astype("string")
df_bronze_layer['variant'] = df_bronze_layer['variant'].astype("string")
df_bronze_layer['full_name'] = df_bronze_layer['full_name'].astype("string")

In [17]:
# check what unit suffixes actually appear in resale_price before writing the parser
def extract_measure_unit(value):
    return re.findall(r'[A-Za-z]+', value)

extracted_units = [unit for value in df_bronze_layer['resale_price'] for unit in extract_measure_unit(value)]
unique_units = list(dict.fromkeys(extracted_units))
print(unique_units)

['Lakh', 'Crore']


Cleaned Resale Price Column:

Converts resale_price to a numeric value in INR.

In [18]:
# "5.45 Lakh" -> "5.45*100000", "10 Crore" -> "10*10000000", then evaluate the
# multiplication; plain numeric strings pass through untouched
df_bronze_layer['cleaned_resale_price'] = df_bronze_layer['resale_price'].replace(
    {"₹": "", " Lakh": "*100000", " Crore": "*10000000"}, regex=True
)
df_bronze_layer['cleaned_resale_price'] = df_bronze_layer['cleaned_resale_price'].str.replace(',', '', regex=True)
# Fixed: used eval(x) on data-derived text as the fallback for plain numeric
# strings - float(x) does the same job without evaluating arbitrary expressions
df_bronze_layer['cleaned_resale_price'] = df_bronze_layer['cleaned_resale_price'].apply(
    lambda x: float(x.split('*')[0]) * float(x.split('*')[1]) if '*' in x else float(x)
)
df_bronze_layer['cleaned_resale_price'] = df_bronze_layer['cleaned_resale_price'].astype(float)
df_bronze_layer['resale_price'] = df_bronze_layer['cleaned_resale_price']
df_bronze_layer.drop(columns=['cleaned_resale_price'], inplace=True)

Engine Capacity:

Converts engine_capacity to numeric value in cc

In [19]:
def remove_cc(val):
    return int(val.replace('cc', '').strip())

df_bronze_layer['engine_capacity'] = df_bronze_layer['engine_capacity'].apply(remove_cc)

KMs Driven:

Converts kms_driven to a numeric value

In [20]:
def remove_kms(val):
    return float(val.replace('Kms', '').replace(',', '').strip())

df_bronze_layer['kms_driven'] = df_bronze_layer['kms_driven'].apply(remove_kms)

Max Power:

Converts max_power to a numeric value in bhp

In [21]:
def extract_power(val):
    match = re.search(r'[\d.]+', val)
    return float(match.group()) if match else None

df_bronze_layer['max_power'] = df_bronze_layer['max_power'].apply(extract_power)

Mileage:

Converts mileage to a numeric value (kmpl)

In [22]:
def extract_mileage(val):
    val = str(val).strip()
    parts = val.split()
    numeric_value = float(parts[0])
    unit = parts[1] if len(parts) > 1 else None

    # Fixed: previously assumed a unit was always present and did `'kmpl' in
    # unit`, which raises a TypeError if unit is None. Doesn't happen on this
    # dataset (missing mileage values are already imputed with a modal string
    # that includes a unit by this point), but it's now guarded either way.
    if unit is None:
        return numeric_value
    if 'kmpl' in unit:
        return numeric_value
    elif 'km/kg' in unit:
        # 1 km/kg = 1.33 kmpl
        return numeric_value * 1.33
    else:
        return numeric_value

df_bronze_layer['mileage'] = df_bronze_layer['mileage'].apply(extract_mileage)

Standardized Transmission:

standardize case and spelling

In [23]:
print(df_bronze_layer['transmission_type'].unique())

<StringArray>
['Manual', 'Automatic']
Length: 2, dtype: string


In [24]:
def standardize_transmission(val):
    val = str(val).strip().lower()
    if val in ['manual', 'm']:
        return 'Manual'
    elif val in ['automatic', 'auto', 'a']:
        return 'Automatic'
    else:
        return 'Unknown'

df_bronze_layer['transmission_type'] = df_bronze_layer['transmission_type'].apply(standardize_transmission)
df_bronze_layer['transmission_type'] = df_bronze_layer['transmission_type'].astype("string")

Owner Type Standardization:

Ensures consistent representation of owner_type

In [25]:
print(df_bronze_layer['owner_type'].unique())

<StringArray>
['First Owner', 'Second Owner', 'Third Owner', 'Fifth Owner', 'Fourth Owner']
Length: 5, dtype: string


In [26]:
def standardize_owner_type(val):
    val = str(val).strip().lower()
    if val in ['first owner', '1st owner']:
        return 'First Owner'
    elif val in ['second owner', '2nd owner']:
        return 'Second Owner'
    elif val in ['third owner', '3rd owner']:
        return 'Third Owner'
    elif val in ['fourth owner', '4th owner']:
        return 'Fourth Owner'
    else:
        return 'Unknown'

df_bronze_layer['owner_type'] = df_bronze_layer['owner_type'].apply(standardize_owner_type)
df_bronze_layer['owner_type'] = df_bronze_layer['owner_type'].astype("string")

In [27]:
df_bronze_layer.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17243 entries, 0 to 17445
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   full_name          17243 non-null  string 
 1   resale_price       17243 non-null  float64
 2   registered_year    17243 non-null  string 
 3   engine_capacity    17243 non-null  int64  
 4   insurance          17243 non-null  string 
 5   transmission_type  17243 non-null  string 
 6   kms_driven         17243 non-null  float64
 7   owner_type         17243 non-null  string 
 8   fuel_type          17243 non-null  string 
 9   max_power          17243 non-null  float64
 10  seats              17243 non-null  float64
 11  mileage            17243 non-null  float64
 12  body_type          17243 non-null  string 
 13  city               17243 non-null  string 
 14  year               17243 non-null  int64  
 15  make               17243 non-null  string 
 16  model              17243 no

Handled data inconsistencies by standardizing column types and removing redundant fields such as 'registered_year' and 'year' to ensure clarity and avoid duplication

In [28]:
df_bronze_layer.drop(columns=['registered_year'], inplace=True)
df_bronze_layer['seats'] = df_bronze_layer['seats'].astype(int)

In [29]:
df_bronze_layer[df_bronze_layer.select_dtypes('string').columns] = (
    df_bronze_layer.select_dtypes('string').apply(lambda x: x.str.strip())
)
df_bronze_layer.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17243 entries, 0 to 17445
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   full_name          17243 non-null  string 
 1   resale_price       17243 non-null  float64
 2   engine_capacity    17243 non-null  int64  
 3   insurance          17243 non-null  string 
 4   transmission_type  17243 non-null  string 
 5   kms_driven         17243 non-null  float64
 6   owner_type         17243 non-null  string 
 7   fuel_type          17243 non-null  string 
 8   max_power          17243 non-null  float64
 9   seats              17243 non-null  int64  
 10  mileage            17243 non-null  float64
 11  body_type          17243 non-null  string 
 12  city               17243 non-null  string 
 13  year               17243 non-null  int64  
 14  make               17243 non-null  string 
 15  model              17243 non-null  string 
 16  variant            17243 no

### 6) Removing and handling outliers
we won't perform any handling to outliers since the upper and lower bound has the same value

In [30]:
Q1 = df_bronze_layer['seats'].quantile(0.25)
Q3 = df_bronze_layer['seats'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(lower_bound)
print(upper_bound)

5.0
5.0


In [31]:
df_silver_layer = df_bronze_layer.copy()
df_silver_layer.to_csv('data/car_silver_layer.csv', index=True)
df_silver_layer.shape

(17243, 17)

# Gold Layer

1.   Feature Engineering
2.   Rename and reorder final dataset

In [32]:
df_silver_layer.columns

Index(['full_name', 'resale_price', 'engine_capacity', 'insurance',
       'transmission_type', 'kms_driven', 'owner_type', 'fuel_type',
       'max_power', 'seats', 'mileage', 'body_type', 'city', 'year', 'make',
       'model', 'variant'],
      dtype='object')

### 1) Feature engineering:

Creating new features enhances data quality and reveals deeper insights in the gold layer by transforming or combining existing columns to highlight key patterns and relationships, supporting better analysis and decision-making

Vehicle Age:

Vehicle Age = Current Year - registered_year

In [33]:
def vehicle_age(registered_year):
    current_year = datetime.datetime.now().year
    return current_year - int(registered_year)

df_silver_layer['vehicle_age'] = df_silver_layer['year'].apply(vehicle_age)

Price per KMs Driven:

Computes ratio of resale price to kilometers driven

In [34]:
df_silver_layer['price_per_km'] = df_silver_layer['resale_price'] / df_silver_layer['kms_driven']

Duplicate Record Flag:

Flags duplicate records based on key vehicle attributes based on full_name, registered_year (now `year`) and kms_driven

In [35]:
dup_mask = df_silver_layer.duplicated(subset=['full_name', 'year', 'kms_driven'], keep=False)
print(dup_mask.value_counts())
df_silver_layer['duplicate_flag'] = dup_mask.astype(int)

False    16147
True      1096
Name: count, dtype: int64


### 2) Rename and reorder final dataset:

Creates a new DataFrame with a specified column order and renamed columns

In [36]:
new_column_order = ['full_name', 'make', 'model', 'variant', 'resale_price', 'year',
                    'engine_capacity', 'insurance', 'transmission_type', 'kms_driven',
                    'owner_type', 'fuel_type', 'max_power', 'seats', 'mileage', 'body_type',
                    'city', 'vehicle_age', 'price_per_km', 'duplicate_flag']

column_rename_map = {
    'engine_capacity': 'engine_capacity_cc',
    'max_power': 'max_power_bhp',
    'mileage': 'mileage_kmpl',
    'duplicate_flag': 'duplicate_record_flag',
    'year': 'registered_year',
}

def create_new_dataframe(df, new_column_order, column_rename_map):
    df_reordered = df[new_column_order]
    df_renamed = df_reordered.rename(columns=column_rename_map)
    return df_renamed

df_final = create_new_dataframe(df_silver_layer, new_column_order, column_rename_map)
df_final.index.name = 'vehicle_id'
df_final.head()

,full_name,make,model,variant,resale_price,registered_year,engine_capacity_cc,insurance,transmission_type,kms_driven,owner_type,fuel_type,max_power_bhp,seats,mileage_kmpl,body_type,city,vehicle_age,price_per_km,duplicate_record_flag
vehicle_id,,,,,,,,,,,,,,,,,,,,
0,2017 Maruti Baleno 1.2 Alpha,Maruti,Baleno,1.2 Alpha,545000.0,2017,1197,Third Party,Manual,40000.0,First Owner,Petrol,83.10,5,21.40,Hatchback,Agra,9,13.625000,0
1,2018 Tata Hexa XTA,Tata,Hexa,XTA,1000000.0,2018,2179,Third Party,Automatic,70000.0,First Owner,Diesel,153.86,7,17.60,MUV,Agra,8,14.285714,0
2,2015 Maruti Swift Dzire VXI,Maruti,Swift,Dzire VXI,450000.0,2015,1197,Third Party,Manual,70000.0,Second Owner,Petrol,83.14,5,20.85,Sedan,Agra,11,6.428571,0
4,2009 Hyundai i10 Magna 1.1,Hyundai,i10,Magna 1.1,160000.0,2009,1086,Third Party,Manual,80000.0,First Owner,Petrol,68.05,5,19.81,Hatchback,Agra,17,2.000000,1
5,2015 Hyundai i20 Active 1.2,Hyundai,i20,Active 1.2,470000.0,2015,1197,Third Party,Manual,70000.0,First Owner,Petrol,81.86,5,17.19,Hatchback,Agra,11,6.714286,0


In [37]:
df_final.to_csv('data/car_gold_layer.csv', index=True)
df_final.shape

(17243, 20)

# Data visualization

Car Resale Dashboard

After preparing the data in this notebook, we moved on to creating the final visualizations in Looker Studio which allowed us to build interactive and insightful charts for business users

Key Elements in the Dashboard:

Scoreboard Metrics: high-level overview of the market

1.  Total Number Of Vehicle
2. Total Number Of Cities
3. Total Number of Models
4. Average of price/KM
5. Median of resale price
6. Median of vehicle age  


Section 1 (Market Composition) : Understand the structure of the car resale market (what's available, what's popular and where listings are coming from)

  - Chart 1 (Pie Chart): Body Type Distribution which Helps identify which types dominate the resale market.

  - Chart 2 (Pie Chart): Fuel Type Distribution which helps to visualize the spread of different fuel types and give insight into engine preferences and eco-conscious trends

  - Chart 3 (Vertical Bar Chart): total vehicle per city to show regional market hotspots for used cars



Section 2 (Pricing & Value Insights) : Understand how resale price is affected mainly by vehicle age and ownership

  - Chart 1 (Line Chart): Average Resale Price by Vehicle Age which shows how vehicle value drops with age

  - Chart 2 (Bar Chart): Avg Price per KM by Owner Type which compares the price per kilometer across ownership types


Interactive Filters:
Added dropdowns to allow users to filter the data dynamically:

  - Body Type – Users can filter by these categories to see how different types of vehicles perform in terms of pricing and value

  - City – Users can also filter by region and age to drill down into more specific data points

Access the Dashboard Here:

https://lookerstudio.google.com/reporting/bbf7122e-257c-4adc-870d-c387e0712271